## Saving verb patterns in database

Saving verb pattern tables *verb_patterns_len1_new.csv* and *verb_patterns_len2_new.csv* from *target_data* directory in a new database.

Database *verb_patterns_new.db* contains two tables: patterns_len1 (one-member patterns) and patterns_len2 (two-member patterns). Table patterns_len1 has following columns:

    ID - pattern ID
    word - verb or verb construction as a string
    government - phrase connected to verb
    verb_word - main verb of verb (construction)
    compound_prt1 - first compound part in a longer verb construction
    compound_prt2 - second compound part in a longer verb construction
    compound_prt3 - third compound part in a longer verb construction
    full_pattern - phrase connected to verb, where question words have been replaced with cases
    w_case - case of the main word in phrase
    adp - adposition in phrase
    verb - infinite verb form in phrase
    other - a member in phrase that does not fit into any of the other categories
    
Table patterns_len2 has following columns:

    ID - pattern ID
    word - verb or verb construction as a string
    government - phrases connected to verb
    verb_word - main verb of verb (construction)
    compound_prt1 - first compound part in a longer verb construction
    compound_prt2 - second compound part in a longer verb construction
    compound_prt3 - third compound part in a longer verb construction
    verb_compound - other parts of a longer verb construction (ordered alphabetically)
    full_pattern - phrase connected to verb, where question words have been replaced with cases
    member1_case - case of the main word in first phrase
    member1_adp - adposition in first phrase
    member1_other - a member in first phrase that does not fit into any of the other categories
    member2_case - case of the main word in second phrase
    member2_adp - adposition in second phrase
    member2_verb - infinite verb form in second phrase
    member2_other - a member in second phrase that does not fit into any of the other categories

In [2]:
import sys

sys.path.append('../../../common_code')

In [3]:
import sqlite3
import pandas as pd
from db_operations.db_display import *

## Input parameters

In [4]:
INPUT_DIR = "../target_data"

RESULT_DB = "verb_patterns_new.db"
INPUT_FILE_LEN1 = f"{INPUT_DIR}/verb_patterns_len1_new.csv"
INPUT_FILE_LEN2 = f"{INPUT_DIR}/verb_patterns_len2_new.csv"

## Data processing

In [5]:
con = sqlite3.connect(RESULT_DB)
cur = con.cursor()
cur.execute('pragma encoding=UTF8')

df_len1_new = pd.read_csv(INPUT_FILE_LEN1)
df_len2_new = pd.read_csv(INPUT_FILE_LEN2)

df_len1_new = df_len1_new.fillna('')
df_len2_new = df_len2_new.fillna('')

new_df1_new = df_len1_new.drop(columns=['Unnamed: 0'])
new_df2_new = df_len2_new.drop(columns=['Unnamed: 0'])

cur.execute("CREATE TABLE patterns_len1(ID INTEGER PRIMARY KEY, word TEXT, government TEXT, verb_word TEXT, compound_prt1 TEXT, compound_prt2 TEXT, compound_prt3 TEXT, full_pattern TEXT, w_case TEXT, adp TEXT, verb TEXT, other TEXT)")
cur.execute("CREATE TABLE patterns_len2(ID INTEGER PRIMARY KEY, word TEXT, government TEXT, verb_word TEXT, compound_prt1 TEXT, compound_prt2 TEXT, compound_prt3 TEXT, full_pattern TEXT, member1_case TEXT, member1_adp TEXT, member1_other TEXT, member2_case TEXT, member2_adp TEXT, member2_verb TEXT, member2_other TEXT)")

for idx, row in new_df1_new.iterrows():
    cur.execute("""INSERT INTO patterns_len1
                          (word, government, verb_word, compound_prt1, compound_prt2, compound_prt3, full_pattern, w_case, adp, verb, other) 
                          VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);""", (row['word'], row['government'], row['verb_word'], row['compound_prt1'], row['compound_prt2'], row['compound_prt3'], row['government_with_case'], row['case'], row['adp'], row['verb'], row['other']))
    con.commit()
    
for idx, row in new_df2_new.iterrows():
    cur.execute("""INSERT INTO patterns_len2
                          (word, government, verb_word, compound_prt1, compound_prt2, compound_prt3, full_pattern, member1_case, member1_adp, member1_other, member2_case, member2_adp, member2_verb, member2_other) 
                          VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);""", (row['word'], row['government'], row['verb_word'], row['compound_prt1'], row['compound_prt2'], row['compound_prt3'], row['government_with_case'], row['member1_case'], row['member1_adp'], row['member1_other'], row['member2_case'], row['member2_adp'], row['member2_verb'], row['member2_other']))
    con.commit()
    
con.close()

## Result

In [6]:
display_sqlite_as_dataframe(RESULT_DB, 'patterns_len1', 10)

,ID,word,government,verb_word,compound_prt1,compound_prt2,compound_prt3,full_pattern,w_case,adp,verb,other
0,1,aasima,keda*,aasima,,,,part,part,,,
1,2,aasima,kelle kallal,aasima,,,,gen kallal,gen,kallal,,
2,3,abielluma,kellega,abielluma,,,,kom,kom,,,
3,4,abikätt ulatama,kellele,ulatama,abikätt,,,all,all,,,
4,5,abstraheeruma,millest/kellest,abstraheeruma,,,,el,el,,,
5,6,adresseerima,mida*,adresseerima,,,,part,part,,,
6,7,adresseerima,kellele,adresseerima,,,,all,all,,,
7,8,aevastama,mille peale,aevastama,,,,gen peale,gen,peale,,
8,9,agiteerima,keda*,agiteerima,,,,part,part,,,
9,10,ahistama,keda*,ahistama,,,,part,part,,,


In [7]:
display_sqlite_as_dataframe(RESULT_DB, 'patterns_len2', 10)

,ID,word,government,verb_word,compound_prt1,compound_prt2,compound_prt3,full_pattern,member1_case,member1_adp,member1_other,member2_case,member2_adp,member2_verb,member2_other
0,1,abistama,keda/mida* + milles,abistama,,,,part + in,part,,,in,,,
1,2,aitama,kellel + mida teha,aitama,,,,ad + part teha,ad,,,part,,teha,
2,3,alla kirjutama,mille/mida,kirjutama,alla,,,part gen,part,,,gen,,,
3,4,ette heitma,kellele/millele + mida*,heitma,ette,,,all + part,all,,,part,,,
4,5,informeerima,keda* + millest,informeerima,,,,part + el,part,,,el,,,
5,6,jagama,mida* + kellega,jagama,,,,part + kom,part,,,kom,,,
6,7,kaasa lööma,milles/kus,lööma,kaasa,,,in ad,in,,,ad,,,
7,8,kaasa tegema,milles/kus,tegema,kaasa,,,in ad,in,,,ad,,,
8,9,kannatama,mille all,kannatama,,,,gen all,gen,,,all,,,
9,10,keelama,kellel + mida teha,keelama,,,,ad + part teha,ad,,,part,,teha,
